# Разведочный анализ (EDA)

**Цель:** понять, как устроена нагрузка, и из каждого наблюдения вывести гипотезу о будущей фиче.
Каждый раздел построен одинаково: **график → что видим → какая фича из этого следует.**

Ядро: **energy-forecast**; запуск: **Run All**.
Файл `aep_clean.csv` (результат `01_load_and_clean.py`) должен лежать в той же папке, что и этот блокнот.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
sns.set_style("whitegrid")

# чистый ряд после 01_load_and_clean.py
df = pd.read_csv("aep_clean.csv", parse_dates=["Datetime"], index_col="Datetime")
y = df["AEP_MW"]
print("период:", df.index.min(), "->", df.index.max(), "| строк:", len(df))
df.head()

## 1. Общий обзор ряда
Смотрим на весь ряд и на пару недель крупным планом.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 7))

y.plot(ax=ax[0], lw=0.3)
ax[0].set(title="Вся история, почасово", ylabel="МВт")

y.loc["2015-01-05":"2015-01-19"].plot(ax=ax[1])
ax[1].set(title="Две недели крупным планом — видны суточные и недельные волны", ylabel="МВт")

plt.tight_layout(); plt.show()

# среднегодовая нагрузка
df.groupby(df.index.year)["AEP_MW"].mean().round(0)

**Что видим.** Резких разрывов и аномалий нет. Среднегодовое потребление колеблется (пик около 2007–2008, дальше лёгкий спад), но выраженного монотонного тренда нет.
**Вывод:** для прогноза на сутки вперёд опираемся на свежие лаги, а не на «трендовую» фичу. На приближённом куске уже глазом видны суточные волны и спад на выходных.

## 2. Суточный профиль
Как нагрузка зависит от часа суток.

In [ ]:
parts = df.copy()
parts["hour"]       = parts.index.hour
parts["dow"]        = parts.index.dayofweek      # 0 = понедельник
parts["month"]      = parts.index.month
parts["is_weekend"] = (parts["dow"] >= 5).astype(int)

hourly = parts.groupby("hour")["AEP_MW"].mean()
hourly.plot(marker="o", figsize=(7, 4), title="Средняя нагрузка по часам суток")
plt.xlabel("час"); plt.ylabel("МВт"); plt.xticks(range(0, 24, 2)); plt.show()

print("пик в", hourly.idxmax(), "ч,  минимум в", hourly.idxmin(), "ч")

**Что видим.** Глубокий ночной провал (минимум ≈ 4 утра), плавный рост днём и вечерний пик около 19 ч. Размах пик/провал — почти 4000 МВт.
**Фича:** `hour` (час суток). Плюс **циклическое** кодирование sin/cos часа, чтобы 23:00 и 00:00 стояли рядом, а не на разных концах шкалы.

## 3. Недельный профиль
Будни против выходных.

In [ ]:
weekly = parts.groupby("dow")["AEP_MW"].mean()
weekly.plot(kind="bar", figsize=(7, 4), title="Средняя нагрузка по дням недели (0 = Пн)")
plt.xlabel("день недели"); plt.ylabel("МВт"); plt.show()

wk = parts.loc[parts.is_weekend == 0, "AEP_MW"].mean()
we = parts.loc[parts.is_weekend == 1, "AEP_MW"].mean()
print(f"будни = {wk:.0f},  выходные = {we:.0f},  ниже на {(wk - we) / wk * 100:.1f}%")

**Что видим.** Будни заметно выше выходных (примерно на 10%), суббота и воскресенье — самые низкие дни. Это спрос предприятий и офисов.
**Фичи:** `dayofweek` и бинарная `is_weekend`.

## 4. Сезонность по месяцам

In [ ]:
monthly = parts.groupby("month")["AEP_MW"].mean()
monthly.plot(marker="o", color="darkorange", figsize=(7, 4), title="Средняя нагрузка по месяцам")
plt.xlabel("месяц"); plt.ylabel("МВт"); plt.xticks(range(1, 13)); plt.show()

**Что видим.** Кривая двугорбая: высокий **зимний** пик (январь — максимум года, отопление) и летний пик (июль–август, кондиционеры), с провалами в апреле и октябре.
**Честная оговорка:** в датасете **нет температуры**, поэтому сезон модель ловит косвенно — через календарь. В реальной системе (и в твоей газовой теме) главный драйвер — именно температура; температура добавлена на следующем этапе (`09_multinode_weather.py`) и дала −25–30 % ошибки.
**Фичи:** `month` и циклический день года (sin/cos от day-of-year).

## 5. Час × месяц — взаимодействие
Меняется ли форма суток от сезона?

In [ ]:
pivot = parts.pivot_table(index="hour", columns="month", values="AEP_MW", aggfunc="mean")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, cmap="rocket_r")
plt.title("Час × месяц, средняя нагрузка (МВт)"); plt.xlabel("месяц"); plt.ylabel("час"); plt.show()

**Что видим.** Форма суток зависит от сезона: зимой **два** пика — утренний (≈ 7–10) и вечерний (≈ 18–22); летом **один** дневной пик (≈ 14–18). То есть `hour` и `month` **взаимодействуют**.
**Вывод для выбора модели:** это сильный аргумент за деревья (CatBoost) — они ловят такие взаимодействия автоматически, без ручного задания.

## 6. Автокорреляция — мост к лаговым фичам
Насколько прошлое предсказывает будущее на разных задержках.

In [ ]:
for lag in [1, 24, 48, 168]:
    print(f"lag {lag:>3} ч: автокорреляция {y.autocorr(lag):.3f}")

plot_acf(y, lags=180, alpha=None)
plt.title("Автокорреляция нагрузки (лаг в часах)"); plt.xlabel("лаг, ч"); plt.show()

**Что видим.** Очень высокая связь на лаге 1 ч (≈ 0.98 — нагрузка инертна), сильные всплески на лаге **24 ч** (≈ 0.87, суточный цикл) и **168 ч** (≈ 0.76, недельный). На графике ACF «горбы» повторяются каждые 24 часа.
**Фичи:** лаги `y(t-24)`, `y(t-48)`, `y(t-168)` и скользящие средние по ним.
**Важно для построения признаков:** для прогноза на 24 ч вперёд нельзя брать лаги короче горизонта (например `y(t-1)`) — на момент прогноза их ещё не существует. Это и есть защита от утечки данных; реализовано в `04_features.py`.

## Итог: гипотезы для признаков
Из разведки получили список того, что будем строить и проверять:
- **Календарь:** `hour`, `dayofweek`, `month`, `is_weekend`
- **Циклические:** sin/cos часа и sin/cos дня года
- **Лаги:** `y(t-24)`, `y(t-48)`, `y(t-168)`
- **Скользящие:** среднее / мин / макс за прошлые сутки и неделю (со сдвигом, без утечки)
- **На будущее:** температура — главный драйвер; в исходном датасете её нет, интегрирована позже (см. `09_multinode_weather.py`)

Далее (`04_features.py`) каждый признак добавляется и проверяется на бэктесте на бэктесте, что он реально улучшает метрику.